# Public Transport Travel Times

Calculate travel times in Public Transport (PT) between two sets of points given a pedestrian network and one or more GTFSs with PT supply (stops, lines, schedules, etc.).

1) Install the needed libraries (`r5py` and `Java` requirement):

In [11]:
!apt-get update -qq
!apt-get install -y openjdk-21-jdk
!pip install r5py

OpenJDK VM warning: the use of signal() and sigset() for signal chaining was deprecated in version 16.0 and will be removed in a future release. Use sigaction() instead.
OpenJDK VM warning: the use of signal() and sigset() for signal chaining was deprecated in version 16.0 and will be removed in a future release. Use sigaction() instead.
OpenJDK VM warning: the use of signal() and sigset() for signal chaining was deprecated in version 16.0 and will be removed in a future release. Use sigaction() instead.
OpenJDK VM warning: the use of signal() and sigset() for signal chaining was deprecated in version 16.0 and will be removed in a future release. Use sigaction() instead.
OpenJDK VM warning: the use of signal() and sigset() for signal chaining was deprecated in version 16.0 and will be removed in a future release. Use sigaction() instead.
OpenJDK VM warning: the use of signal() and sigset() for signal chaining was deprecated in version 16.0 and will be removed in a future release. Use s

In [12]:
!java -version

[0.001s][warning][os,container] Cgroup memory controller path at '/sys/fs/cgroup' seems to have moved to '/../../jupyter-children', detected limits won't be accurate
[0.001s][warning][os,container] Cgroup cpu controller path at '/sys/fs/cgroup' seems to have moved to '/../../jupyter-children', detected limits won't be accurate
openjdk version "21.0.10" 2026-01-20
OpenJDK Runtime Environment (build 21.0.10+7-Ubuntu-122.04)
OpenJDK 64-Bit Server VM (build 21.0.10+7-Ubuntu-122.04, mixed mode, sharing)


2) Download the needed data

2.1.) Download the pedestrian network (access to stations)

In [13]:
!wget -O madrid.osm.pbf https://download.geofabrik.de/europe/spain/madrid-latest.osm.pbf

OpenJDK VM warning: the use of signal() and sigset() for signal chaining was deprecated in version 16.0 and will be removed in a future release. Use sigaction() instead.
--2026-05-14 11:58:33--  https://download.geofabrik.de/europe/spain/madrid-latest.osm.pbf
Resolving download.geofabrik.de (download.geofabrik.de)... 95.216.245.185, 95.217.45.61, 95.217.63.98, ...
Connecting to download.geofabrik.de (download.geofabrik.de)|95.216.245.185|:443... connected.
HTTP request sent, awaiting response... 302 Found
Location: https://download.geofabrik.de/europe/spain/madrid-260512.osm.pbf [following]
--2026-05-14 11:58:33--  https://download.geofabrik.de/europe/spain/madrid-260512.osm.pbf
Reusing existing connection to download.geofabrik.de:443.
HTTP request sent, awaiting response... 200 OK
Length: 83128687 (79M) [application/octet-stream]
Saving to: ‘madrid.osm.pbf’

madrid.osm.pbf      100%[===================>]  79.28M  14.1MB/s    in 6.6s    

2026-05-14 11:58:40 (12.0 MB/s) - ‘madrid.osm.p

2.2.) Download the GTFSs (PT lines, stops and schedules)

We can download as many GTFSs as we need.

First, we download EMT local bus supply and save it as `gtfs_emt.zip`:

In [14]:
!wget -O gtfs_emt.zip "https://datos.emtmadrid.es/dataset/9b23259a-4491-494b-9695-36a7709b2c12/resource/3cba2058-9833-422c-a704-bf992d31d2ee/download/gtfs_emt.zip"

OpenJDK VM warning: the use of signal() and sigset() for signal chaining was deprecated in version 16.0 and will be removed in a future release. Use sigaction() instead.
--2026-05-14 11:58:40--  https://datos.emtmadrid.es/dataset/9b23259a-4491-494b-9695-36a7709b2c12/resource/3cba2058-9833-422c-a704-bf992d31d2ee/download/gtfs_emt.zip
Resolving datos.emtmadrid.es (datos.emtmadrid.es)... 195.57.112.136
Connecting to datos.emtmadrid.es (datos.emtmadrid.es)|195.57.112.136|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 21039903 (20M) [application/zip]
Saving to: ‘gtfs_emt.zip’

gtfs_emt.zip        100%[===================>]  20.06M  5.79MB/s    in 3.5s    

2026-05-14 11:58:45 (5.79 MB/s) - ‘gtfs_emt.zip’ saved [21039903/21039903]



Now, we download metro supply and save it as `gtfs_metro.zip`:

In [15]:
!wget -O gtfs_metro.zip "https://crtm.maps.arcgis.com/sharing/rest/content/items/5c7f2951962540d69ffe8f640d94c246/data"

OpenJDK VM warning: the use of signal() and sigset() for signal chaining was deprecated in version 16.0 and will be removed in a future release. Use sigaction() instead.
--2026-05-14 11:58:45--  https://crtm.maps.arcgis.com/sharing/rest/content/items/5c7f2951962540d69ffe8f640d94c246/data
Resolving crtm.maps.arcgis.com (crtm.maps.arcgis.com)... 3.163.189.84, 3.163.189.91, 3.163.189.5, ...
Connecting to crtm.maps.arcgis.com (crtm.maps.arcgis.com)|3.163.189.84|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 1503773 (1.4M) [application/zip]
Saving to: ‘gtfs_metro.zip’

gtfs_metro.zip      100%[===================>]   1.43M  5.02MB/s    in 0.3s    

2026-05-14 11:58:46 (5.02 MB/s) - ‘gtfs_metro.zip’ saved [1503773/1503773]



3) Create the origin and destination point datasets

In [16]:
import geopandas as gpd
import pandas as pd

origins_df = pd.DataFrame(
    {
        "id": ["atocha", "sol", "moncloa"],
        "lon": [-3.6896, -3.7038, -3.7194],
        "lat": [40.4066, 40.4169, 40.4352],
    }
)

destinations_df = pd.DataFrame(
    {
        "id": ["nuevos_ministerios", "chamartin", "airport_t4"],
        "lon": [-3.6922, -3.6821, -3.5933],
        "lat": [40.4466, 40.4722, 40.4918],
    }
)

origins_gdf = gpd.GeoDataFrame(
    origins_df,
    geometry=gpd.points_from_xy(origins_df["lon"], origins_df["lat"]),
    crs="EPSG:4326",
)

destinations_gdf = gpd.GeoDataFrame(
    destinations_df,
    geometry=gpd.points_from_xy(destinations_df["lon"], destinations_df["lat"]),
    crs="EPSG:4326",
)

4) Calculate the trip duration matrix

In [17]:
import datetime
import geopandas as gpd
import r5py

transport_network = r5py.TransportNetwork(
    "madrid.osm.pbf",
    ["gtfs_emt.zip", "gtfs_metro.zip"],   # we can add as many GTFS as we want here, with the names that we have given to each when we downloaded them
)

matrix = r5py.TravelTimeMatrix(
    transport_network,
    origins=origins_gdf,
    destinations=destinations_gdf,
    transport_modes=[r5py.TransportMode.TRANSIT],
    departure=datetime.datetime(2026, 5, 14, 8, 0)
)

matrix

GtfsFileError: Could not load GTFS file gtfs_metro.zip. 
EmptyTableError: fare_attributes line 0: Table is present in zip file, but it has no entries.
- EmptyTableError: fare_rules line 0: Table is present in zip file, but it has no entries.